# ICOIN Paper Figures (IEEE style)
실험 결과 그림을 재현하는 노트북. 데이터셋 없이 결과 숫자만으로 그려집니다.
셀을 위에서부터 실행하세요. 각 그림은 600dpi PNG로 저장되고, 마지막 셀에서 한꺼번에 내려받을 수 있습니다.

- 색: Okabe-Ito 색맹 안전 팔레트
- 흑백 인쇄 대비: 마커/빗금(hatch)으로 구분
- 숫자를 바꾸면(아래 '결과 값' 셀) 그림이 자동으로 갱신됩니다.

In [ ]:
# ===== 0. 설정 & IEEE 스타일 =====
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams

rcParams.update({
    'font.family':'serif', 'font.serif':['DejaVu Serif'],
    'font.size':9, 'axes.labelsize':9, 'axes.titlesize':9,
    'xtick.labelsize':8, 'ytick.labelsize':8, 'legend.fontsize':7.5,
    'axes.linewidth':0.7, 'lines.linewidth':1.3, 'savefig.dpi':600,
    'savefig.bbox':'tight', 'axes.grid':True, 'grid.alpha':0.25,
    'grid.linewidth':0.5, 'legend.framealpha':0.95, 'legend.edgecolor':'0.7'
})
# Okabe-Ito colorblind-safe
C_INSDN='#0072B2'   # blue
C_CIC  ='#D55E00'   # vermillion
GRAY   ='#9a9a9a'
print('style ready')

In [ ]:
# ===== 1. 결과 값 (여기 숫자만 바꾸면 그림이 갱신됨) =====
# InSDN: experts 1..5 (N=5),  CIC-DDoS2019: experts 1..4 (N=4)
insdn_k = [1,2,3,4,5];  insdn_f1 = [0.862, 0.886, 0.940, 0.941, 0.941]
cic_k   = [1,2,3,4];    cic_f1   = [0.901, 0.917, 0.917, 0.917]
adaptive_insdn = (2.50, 0.940)   # (avg experts, F1)
adaptive_cic   = (2.50, 0.917)

# 모델 비교 (F1-macro)
models = ['Rule','DT','RF','MoE\n(top-2)','MoE\n(top-3)']
f1_insdn = [0.276, 0.981, 0.977, 0.886, 0.940]
f1_cic   = [np.nan, 0.918, 0.908, 0.917, 0.917]

# 라우팅 정확도 (InSDN, 클래스별)
route_cls = ['DDoS','Probe','Normal','DoS','Other']
route_acc = [0.9998, 0.998, 0.986, 0.938, 0.243]
route_overall = 0.983

# 룰 설치 지연 (ms) — events.csv의 14건
latency = [0.08,0.07,0.07,0.07,0.07,0.07,0.08,0.21,0.20,0.08,0.08,0.19,0.19]
latency_mean = 0.118
print('values loaded')

In [ ]:
# ===== 2. Fig: Load-Adaptive Top-k Pareto (핵심 그림) =====
fig,ax=plt.subplots(figsize=(3.45,2.55))
ax.plot(insdn_k,insdn_f1,'o-',color=C_INSDN,ms=4,lw=1.2,label='InSDN (fixed $k$)',zorder=2)
ax.plot(cic_k,cic_f1,'s-',color=C_CIC,ms=4,lw=1.2,label='CIC-DDoS2019 (fixed $k$)',zorder=2)
ax.scatter([adaptive_insdn[0]],[adaptive_insdn[1]],marker='*',s=180,color=C_INSDN,edgecolor='black',lw=0.6,zorder=5)
ax.scatter([adaptive_cic[0]],[adaptive_cic[1]],marker='*',s=180,color=C_CIC,edgecolor='black',lw=0.6,zorder=5)
# -50% compute 주석 (InSDN: full=5 -> adaptive=2.5)
yv=adaptive_insdn[1]-0.0015
ax.annotate('', xy=(adaptive_insdn[0],yv), xytext=(insdn_k[-1],yv),
            arrowprops=dict(arrowstyle='<->',color=GRAY,lw=0.9))
save=100*(insdn_k[-1]-adaptive_insdn[0])/insdn_k[-1]
ax.text((adaptive_insdn[0]+insdn_k[-1])/2, yv-0.004, f'$-{save:.0f}\\%$ compute',
        ha='center',va='top',fontsize=7,color='0.3')
ax.text(adaptive_insdn[0],adaptive_insdn[1]+0.005,'adaptive',ha='center',fontsize=7,color=C_INSDN)
ax.set_xlabel('Average experts executed (compute)')
ax.set_ylabel('F1-macro'); ax.set_xticks([1,2,3,4,5]); ax.set_ylim(0.85,0.95)
ax.legend(loc='lower right')
fig.savefig('fig_adaptive_tradeoff.png'); plt.show()

In [ ]:
# ===== 3. Fig: 모델별 탐지 성능 (두 데이터셋) =====
fig,ax=plt.subplots(figsize=(3.45,2.55))
x=np.arange(len(models)); w=0.38
ax.bar(x-w/2,f1_insdn,w,label='InSDN',color=C_INSDN,edgecolor='black',lw=0.4)
ax.bar(x+w/2,[0 if np.isnan(v) else v for v in f1_cic],w,label='CIC-DDoS2019',
       color=C_CIC,edgecolor='black',lw=0.4,hatch='///')
ax.set_ylabel('F1-macro'); ax.set_ylim(0,1.05)
ax.set_xticks(x); ax.set_xticklabels(models)
ax.axhline(0.94,ls=':',color=GRAY,lw=0.8)
ax.legend(loc='lower left')
fig.savefig('fig_detect.png'); plt.show()

In [ ]:
# ===== 4. Fig: 클래스별 라우팅 정확도 (InSDN) =====
fig,ax=plt.subplots(figsize=(3.45,2.3))
colors=[C_INSDN]*4+[GRAY]
ax.barh(range(len(route_cls)),route_acc,color=colors,edgecolor='black',lw=0.4)
ax.set_yticks(range(len(route_cls))); ax.set_yticklabels(route_cls)
ax.invert_yaxis(); ax.set_xlabel('Routing accuracy'); ax.set_xlim(0,1.05)
for i,v in enumerate(route_acc): ax.text(min(v+0.02,0.9),i,f'{v:.3f}',va='center',fontsize=7)
ax.axvline(route_overall,ls=':',color='0.4',lw=0.8)
ax.text(route_overall,-0.6,f'overall {route_overall:.3f}',fontsize=6.5,color='0.4',ha='center')
fig.savefig('fig_routing.png'); plt.show()

In [ ]:
# ===== 5. Fig: 룰 설치 지연 히스토그램 =====
fig,ax=plt.subplots(figsize=(3.45,2.3))
ax.hist(latency,bins=8,color=C_INSDN,edgecolor='black',lw=0.5,alpha=0.85)
ax.axvline(latency_mean,ls='--',color=C_CIC,lw=1.2,label=f'mean {latency_mean:.3f} ms')
ax.set_xlabel('Rule-installation latency (ms)'); ax.set_ylabel('Count')
ax.legend(loc='upper right')
fig.savefig('fig_latency.png'); plt.show()

In [ ]:
# ===== 6. (Colab) 그림 한꺼번에 내려받기 =====
try:
    from google.colab import files
    for f in ['fig_adaptive_tradeoff.png','fig_detect.png','fig_routing.png','fig_latency.png']:
        files.download(f)
except Exception as e:
    print('Colab 환경이 아니면 현재 폴더에 PNG가 저장되어 있습니다:', e)